In [1]:
""" This modules applies the final decisions of pre-processing made in 'before_pre-processing.ipynb' directory. 

Corrected Strategy (Audit-Verified):
1. Create _Measured flags for ALL features (including those about to be dropped).
2. Drop low-utility features (keep their _Measured flags).
3. Sort by [Patient_ID, ICULOS] to prevent temporal leakage.
4. Apply physiological limit outlier replacement -> NaN.
5. Apply cross-validation outlier replacement (BP, Hgb/Hct) -> NaN.
6. Patient-level stratified train/test split.
7. LOCF with staleness limit (train only, apply to test).
8. BOCF with short window for early-stay rows.
9. Compute global medians from TRAINING data only.
10. Engineer temporal features (deltas, rolling stats).
11. Compute clinical scores (qSOFA, SIRS, organ dysfunction flags).
12. Validate: zero nulls, BP constraint, physiological bounds.
13. Export train and test as separate files.
"""

" This modules applies the final decisions of pre-processing made in 'before_pre-processing.ipynb' directory. \n\nCorrected Strategy (Audit-Verified):\n1. Create _Measured flags for ALL features (including those about to be dropped).\n2. Drop low-utility features (keep their _Measured flags).\n3. Sort by [Patient_ID, ICULOS] to prevent temporal leakage.\n4. Apply physiological limit outlier replacement -> NaN.\n5. Apply cross-validation outlier replacement (BP, Hgb/Hct) -> NaN.\n6. Patient-level stratified train/test split.\n7. LOCF with staleness limit (train only, apply to test).\n8. BOCF with short window for early-stay rows.\n9. Compute global medians from TRAINING data only.\n10. Engineer temporal features (deltas, rolling stats).\n11. Compute clinical scores (qSOFA, SIRS, organ dysfunction flags).\n12. Validate: zero nulls, BP constraint, physiological bounds.\n13. Export train and test as separate files.\n"

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv("Datasets\\raw\\Dataset.csv")

In [4]:
"""
Removing Features:

1. Hour & Unnamed: 0
These are simply sequential row counters generated when the individual patient files were merged into a single CSV. 
The true medical metric for time elapsed in the ICU is ICULOS, making these columns mathematically redundant.

2. Unit2
This is a binary indicator for the ICU unit that is perfectly inversely correlated with Unit1 (if one is 1, the other is 0). 
Keeping both causes multicollinearity in machine learning models, so we drop this and keep Unit1.

3. SaO2
Arterial oxygen saturation is ~96.5% missing because it requires a painful arterial blood draw. 
We already have O2Sat from the continuous finger pulse oximeter (~13% missing), which provides the exact same oxygenation signal.

4. BaseExcess
This is ~94.5% missing and clinically redundant for a baseline model. 
The patient's metabolic and acid-base balance is already perfectly captured by the core features we kept: pH, PaCO2, and HCO3

5. EtCO2
Missing in ~96.3% of rows because it is primarily monitored only when patients are intubated and on a mechanical ventilator. 
Respiratory distress is already well-covered by our Resp and O2Sat features without introducing this massive sparsity.

6. TroponinI
This is a highly specific biomarker for heart attacks (myocardial infarction), missing in ~99.0% of rows. 
It is too sparse to be useful for a generalized baseline sepsis model, and imputing it would just create noise.

7. Bilirubin_direct, Bilirubin_total, AST, Alkalinephos
These liver enzymes are >98% missing across the dataset. While liver dysfunction can happen in severe sepsis, 
imputing columns that are nearly 100% empty will just create flat, useless data arrays for the model.

8. Fibrinogen & PTT
These secondary coagulation markers are >97% missing. We are already keeping Platelets, 
which is the primary and most important indicator for sepsis-induced blood clotting issues (DIC), 
making these redundant and too sparse to keep.

Total features dropped = 13
"""

"\nRemoving Features:\n\n1. Hour & Unnamed: 0\nThese are simply sequential row counters generated when the individual patient files were merged into a single CSV. \nThe true medical metric for time elapsed in the ICU is ICULOS, making these columns mathematically redundant.\n\n2. Unit2\nThis is a binary indicator for the ICU unit that is perfectly inversely correlated with Unit1 (if one is 1, the other is 0). \nKeeping both causes multicollinearity in machine learning models, so we drop this and keep Unit1.\n\n3. SaO2\nArterial oxygen saturation is ~96.5% missing because it requires a painful arterial blood draw. \nWe already have O2Sat from the continuous finger pulse oximeter (~13% missing), which provides the exact same oxygenation signal.\n\n4. BaseExcess\nThis is ~94.5% missing and clinically redundant for a baseline model. \nThe patient's metabolic and acid-base balance is already perfectly captured by the core features we kept: pH, PaCO2, and HCO3\n\n5. EtCO2\nMissing in ~96.3% 

In [5]:
# 1. Sort by Patient_ID and ICULOS to prevent temporal leakage (Addresses Flaw 4)
df = df.sort_values(['Patient_ID', 'ICULOS']).reset_index(drop=True)

# 2. Apply Physiological Limits (Addresses Flaw 7: Consistent HR bounds)
vital_limits = {
    'HR': (30.0, 200.0),   # Corrected to match EDA and clinical standards
    'MAP': (30.0, 200.0),
    'SBP': (40.0, 250.0),
    'DBP': (20.0, 160.0),
    'O2Sat': (50.0, 100.0),
    'Resp': (4.0, 60.0),
    'Temp': (24.0, 43.0)
}

lab_limits = {
    'HCO3': (2.0, 60.0), 'FiO2': (0.20, 1.0), 'pH': (6.5, 8.0), 'PaCO2': (5.0, 120.0),
    'BUN': (1.0, 300.0), 'Calcium': (1.0, 20.0), 'Chloride': (20.0, 160.0),
    'Creatinine': (0.1, 30.0), 'Glucose': (10.0, 1500.0), 'Lactate': (0.1, 30.0),
    'Magnesium': (0.1, 15.0), 'Phosphate': (0.1, 20.0), 'Potassium': (1.0, 10.0),
    'Hct': (5.0, 75.0), 'Hgb': (2.0, 30.0), 'WBC': (0.1, 300.0), 'Platelets': (1.0, 3000.0)
}

for col, (min_val, max_val) in {**vital_limits, **lab_limits}.items():
    if col in df.columns:
        df[col] = df[col].where((df[col] >= min_val) & (df[col] <= max_val), np.nan)

# 3. Cross-Validation Outlier Replacement (Addresses Flaw 10: Tight Hgb/Hct ratio)
# Fix impossible Blood Pressures
invalid_bp = (df['SBP'] < df['MAP']) | (df['MAP'] < df['DBP']) | (df['SBP'] < df['DBP'])
df.loc[invalid_bp, ['SBP', 'MAP', 'DBP']] = np.nan

# Fix impossible Hgb/Hct ratios (Rule of Three: Hct ≈ 3x Hgb)
both_present = df['Hgb'].notna() & df['Hct'].notna()
ratio = df.loc[both_present, 'Hct'] / df.loc[both_present, 'Hgb']
invalid_blood = both_present & ((ratio < 2.5) | (ratio > 4.0))
df.loc[invalid_blood, ['Hct', 'Hgb']] = np.nan

print(f"Initial cleaning complete. Impossible values nulled and dataset sorted.")

Initial cleaning complete. Impossible values nulled and dataset sorted.


In [6]:
# 1. Drop the anomalous Patient 13777 (HospAdmTime outlier)
# We use .copy() to prevent Pandas from throwing a "SettingWithCopy" warning later
df = df[df['Patient_ID'] != 13777].copy()

# 2. Refactor Unit1 (Addresses Flaw 9)
if 'Unit1' in df.columns:
    # Create missingness flag
    df['Unit1_Unknown'] = df['Unit1'].isna().astype(int)
    # Fill missing with baseline category 0 (not MICU) instead of -1
    df['Unit1'] = df['Unit1'].fillna(0).astype(int)

print(f"Patient 13777 removed. New dataset shape: {df.shape}")
print("Unit1 refactored: missing values flagged and filled with 0.")


Patient 13777 removed. New dataset shape: (1552202, 45)
Unit1 refactored: missing values flagged and filled with 0.


In [7]:
# 1. Define ALL clinical features (Vitals + Labs)
vitals = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp']
labs = ['HCO3', 'FiO2', 'pH', 'PaCO2', 'BUN', 'Calcium', 'Chloride', 'Creatinine', 
        'Glucose', 'Lactate', 'Magnesium', 'Phosphate', 'Potassium', 'Hct', 'Hgb', 
        'WBC', 'Platelets']
features_to_drop_names = ['Bilirubin_direct', 'Bilirubin_total', 'AST', 'Alkalinephos', 'Fibrinogen', 
                          'PTT', 'TroponinI', 'EtCO2', 'BaseExcess', 'SaO2']

all_clinical_cols = vitals + labs + features_to_drop_names

# 2. Create _Measured flags for ALL clinical features based on cleaned state (Addresses Flaw 1 & 8)
for col in all_clinical_cols:
    if col in df.columns:
        df[f"{col}_Measured"] = df[col].notna().astype(int)

# 3. Drop low-utility raw features (keep their flags)
admin_cols = ['Unit2', 'Hour', 'Unnamed: 0']
df = df.drop(columns=features_to_drop_names + admin_cols, errors='ignore')

print(f"Measurement flags created for {len(all_clinical_cols)} features.")
print(f"Low-utility raw features dropped. New shape: {df.shape}")

Measurement flags created for 34 features.
Low-utility raw features dropped. New shape: (1552202, 66)


In [8]:
# 6. Patient-level Stratified Train/Test Split (Addresses Flaw 3 & 14)
from sklearn.model_selection import StratifiedGroupKFold

# Get patient-level label (1 if the patient EVER developed sepsis)
patient_labels = df.groupby('Patient_ID')['SepsisLabel'].max()

# Use StratifiedGroupKFold for stratified patient-level splitting
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# Get one fold as test (20%), rest as train (80%)
for train_patients, test_patients in sgkf.split(
    df, y=patient_labels.loc[df['Patient_ID']].values, groups=df['Patient_ID']
):
    break

train_df = df.iloc[train_patients].copy().reset_index(drop=True)
test_df  = df.iloc[test_patients].copy().reset_index(drop=True)

print(f"Data split into Train: {train_df.shape} and Test: {test_df.shape}")


Data split into Train: (1241761, 66) and Test: (310441, 66)


In [9]:
# 7. & 8. Temporal Imputation: LOCF and BOCF (Addresses Flaw 5 & 6)
# Explicitly list the administrative and static columns to ignore
ignore_cols = ['Patient_ID', 'Age', 'Gender', 'Unit1', 'Unit1_Unknown', 'HospAdmTime', 'ICULOS', 'SepsisLabel']

# Dynamically select ONLY the Vitals and Labs
cols_to_impute = [col for col in train_df.columns if col not in ignore_cols and not col.endswith('_Measured')]
vital_cols = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp']
lab_cols = [c for c in cols_to_impute if c not in vital_cols]

def apply_temporal_imputation(data):
    # Vitals measured hourly — LOCF up to 4 hours
    data[vital_cols] = data.groupby('Patient_ID')[vital_cols].ffill(limit=4)
    # Lab values drawn less frequently — LOCF up to 24 hours
    data[lab_cols] = data.groupby('Patient_ID')[lab_cols].ffill(limit=24)
    
    # BOCF: backward fill for the early rows that LOCF couldn't reach
    data[cols_to_impute] = data.groupby('Patient_ID')[cols_to_impute].bfill(limit=6)
    return data

train_df = apply_temporal_imputation(train_df)
test_df = apply_temporal_imputation(test_df)

print("Temporal imputation (LOCF & BOCF) complete on both splits.")


Temporal imputation (LOCF & BOCF) complete on both splits.


In [10]:
# 9. Compute global medians from TRAINING data only (Addresses Flaw 2 & 11)
train_medians = train_df[cols_to_impute].median()

# Adjust FiO2 fallback to clinical conditional median instead of room air (Flaw 11)
fio2_fallback = train_df.loc[train_df['FiO2'].notna(), 'FiO2'].median()
train_medians['FiO2'] = fio2_fallback

# Apply medians to both splits
train_df[cols_to_impute] = train_df[cols_to_impute].fillna(train_medians)
test_df[cols_to_impute]  = test_df[cols_to_impute].fillna(train_medians)

print("Global fallback imputation complete (using train-set statistics).")


Global fallback imputation complete (using train-set statistics).


In [11]:
# Fix BP constraint after imputation (Flaw 15)
def enforce_bp_constraint(data):
    data["DBP"] = __import__("numpy").minimum(data["DBP"], data["MAP"])
    data["SBP"] = __import__("numpy").maximum(data["SBP"], data["MAP"])
    return data

train_df = enforce_bp_constraint(train_df)
test_df = enforce_bp_constraint(test_df)
print("BP constraint enforced post-imputation.")


BP constraint enforced post-imputation.


In [12]:
# 10. Engineer temporal features (deltas, rolling stats) - Flaw 12
def engineer_temporal_features(data):
    key_vitals = ['HR', 'MAP', 'SBP', 'Resp', 'O2Sat', 'Temp']
    key_labs   = ['Lactate', 'Creatinine', 'WBC', 'Glucose', 'Platelets']
    
    for col in key_vitals + key_labs:
        grp = data.groupby('Patient_ID')[col]
        # Change from 1 hour ago
        data[f'{col}_delta_1h'] = grp.diff(periods=1)
        # Change from 6 hours ago
        data[f'{col}_delta_6h'] = grp.diff(periods=6)
        # 6-hour rolling mean
        data[f'{col}_roll6_mean'] = grp.transform(lambda x: x.rolling(6, min_periods=1).mean())
        # 6-hour rolling std (volatility)
        data[f'{col}_roll6_std']  = grp.transform(lambda x: x.rolling(6, min_periods=1).std())
        
        # Fill missing early deltas/stds with 0 (assuming stable if missing)
        data[f'{col}_delta_1h'] = data[f'{col}_delta_1h'].fillna(0)
        data[f'{col}_delta_6h'] = data[f'{col}_delta_6h'].fillna(0)
        data[f'{col}_roll6_std'] = data[f'{col}_roll6_std'].fillna(0)
        
    return data

train_df = engineer_temporal_features(train_df)
test_df = engineer_temporal_features(test_df)
print("Temporal feature engineering complete.")


Temporal feature engineering complete.


In [13]:
# 11. Compute clinical scores (qSOFA, SIRS, organ dysfunction flags) - Flaw 13
def compute_clinical_scores(data):
    # qSOFA components (Sepsis-3)
    data['qSOFA_SBP']  = (data['SBP'] <= 100).astype(int)
    data['qSOFA_Resp'] = (data['Resp'] >= 22).astype(int)
    data['qSOFA_score'] = data['qSOFA_SBP'] + data['qSOFA_Resp']
    
    # SIRS criteria
    data['SIRS_Temp'] = ((data['Temp'] < 36) | (data['Temp'] > 38)).astype(int)
    data['SIRS_HR']   = (data['HR'] > 90).astype(int)
    data['SIRS_Resp'] = (data['Resp'] > 20).astype(int)
    data['SIRS_WBC']  = ((data['WBC'] < 4) | (data['WBC'] > 12)).astype(int)
    data['SIRS_score'] = data[['SIRS_Temp', 'SIRS_HR', 'SIRS_Resp', 'SIRS_WBC']].sum(axis=1)
    
    # Organ dysfunction markers (SOFA components)
    data['renal_flag']  = (data['Creatinine'] >= 1.2).astype(int)
    data['coag_flag']   = (data['Platelets'] < 150).astype(int)
    data['shock_flag']  = (data['MAP'] < 65).astype(int)
    data['hypoxia_flag'] = (data['O2Sat'] < 94).astype(int)
    data['acidosis_flag'] = (data['pH'] < 7.35).astype(int)
    
    return data

train_df = compute_clinical_scores(train_df)
test_df = compute_clinical_scores(test_df)
print("Clinical score computation complete.")


Clinical score computation complete.


In [14]:
# 12. Post-Imputation Validation Assertions (Flaw 15)
def validate_data(data, name="Train"):
    print(f"=== POST-IMPUTATION VALIDATION: {name} ===")
    
    # 1. No nulls in clinical features
    null_counts = data[cols_to_impute].isnull().sum()
    assert null_counts.sum() == 0, f"Remaining nulls in {name}:\n{null_counts[null_counts > 0]}"
    
    # 2. BP constraint holds
    bp_valid = (data['SBP'] >= data['MAP']) & (data['MAP'] >= data['DBP'])
    assert bp_valid.all(), f"BP constraint violated in {(~bp_valid).sum()} rows after imputation in {name}"
    
    # 3. No constant features
    std_check = data[cols_to_impute].std()
    constant_cols = std_check[std_check == 0].index.tolist()
    assert not constant_cols, f"Constant features after imputation in {name}: {constant_cols}"
    
    print(f"All assertions passed for {name}.\n")

validate_data(train_df, "Train")
validate_data(test_df, "Test")


=== POST-IMPUTATION VALIDATION: Train ===


All assertions passed for Train.

=== POST-IMPUTATION VALIDATION: Test ===
All assertions passed for Test.



In [15]:
# 13. Export train and test as separate files
import os

if not os.path.exists('Datasets/processed'):
    os.makedirs('Datasets/processed')

train_df.to_csv('Datasets/processed/sepsis_icu_train.csv', index=False)
test_df.to_csv('Datasets/processed/sepsis_icu_test.csv', index=False)

print("Train and Test datasets successfully exported! Safe to close the notebook.")


Train and Test datasets successfully exported! Safe to close the notebook.
